In [1]:
from __future__ import annotations
import os
import json
import importlib
import pprint
import time

import random
import numpy as np
np.set_printoptions(4)

import torch

from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL)

# from core import mcts_search_test_v21
# from core import mcts_search_test_v31
# from core import mcts_search_test_v61
from core import mcts_embeds_search_v03_02_00

from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k_hf
from utils.utils import add_completions_to_dataset_tree


# algo_dict = {
#     "mcts_embeds": mcts_embeds_search_v03_02_00
# }

# algo_version = "v03_02_00"
# algo_name = "mcts_embeds"
# algo = algo_dict[algo_name]

INFO 11-04 09:14:04 [__init__.py:244] Automatically detected platform cuda.


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits1"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
llm_total_gpu = 0.4
llm_gpu_memory_utilization = 0.2
# llm_vllm = LLM(
#     model = llm_dir,
#     tensor_parallel_size=1,
#     gpu_memory_utilization = 0.7,  # Utilize 50% of GPU memory
#     # enable_prefix_caching=True,  # V100 doesn't support enable_prefix_caching 
#     # enable_chunked_prefill=False, # and enable_chunked_prefill
#     max_model_len = 5000,
#     dtype = "float16",
#     seed = config.seed)

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)

print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))


INFO 11-04 09:14:23 [config.py:823] This model supports multiple tasks: {'embed', 'classify', 'reward', 'generate', 'score'}. Defaulting to 'generate'.
WARNING 11-04 09:14:23 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 11-04 09:14:23 [arg_utils.py:1642] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 11-04 09:14:23 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-04 09:14:23 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_par

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 11-04 09:14:26 [default_loader.py:272] Loading weights took 1.27 seconds
INFO 11-04 09:14:27 [model_runner.py:1203] Model loading took 2.3185 GiB and 1.385294 seconds
INFO 11-04 09:14:28 [worker.py:294] Memory profiling takes 0.53 seconds
INFO 11-04 09:14:28 [worker.py:294] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.20) = 6.35GiB
INFO 11-04 09:14:28 [worker.py:294] model weights take 2.32GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 2.75GiB.
INFO 11-04 09:14:28 [executor_base.py:113] # cuda blocks: 5631, # CPU blocks: 32768
INFO 11-04 09:14:28 [executor_base.py:118] Maximum concurrency for 5000 tokens per request: 18.02x
INFO 11-04 09:14:34 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 7.15 seconds
#--- memory: 5.07647705078125


In [5]:
llm_vllm_embeds = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    task="embed",
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

WARNING 11-04 09:14:34 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 11-04 09:14:34 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-04 09:14:34 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_pr

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 11-04 09:14:37 [default_loader.py:272] Loading weights took 1.26 seconds
INFO 11-04 09:14:37 [model_runner.py:1203] Model loading took 2.3029 GiB and 1.306664 seconds
#--- memory: 7.379390716552734


In [6]:
prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

#--- memory: 22.336918354034424


In [7]:
stop

NameError: name 'stop' is not defined

In [8]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 20            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0
config.date_string = "Aug 1 2025"

config.num_batches = 2
config.step_budget = config.num_batches*config.max_depths 
config.num_phases = 100

config.lam = 0.01
config.embeds_normalizing = True
config.embeds_centering = True
config.embeds_mean_dir = "embeds_mean_bon--level-4--v01_0_0--bs-256"

config.cpuct = 2
config.ds_beta = 1.0
config.ds_alpha = 100.0
config.use_ppl = True
config.negative_reward = 0

config.version = "t811"

In [16]:

#  load data 
level = 4                                   # level of difficulty of questions
num_trials = 5
print(f"num_trials = {num_trials}")

print(data_dir)
# data_by_levels = load_data_prm800k(data_dir)
dataset = load_data_prm800k_hf(data_dir, level)

num_questions = len(dataset)  # level 4 has 128 questions
num_questions = 2
print(f"num_questions = {num_questions}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
batch_of_questions = [dataset[idx]['problem'] for idx in range(num_questions)]

if config.embeds_centering:
    config.embeds_mean = np.load(f"results/{config.embeds_mean_dir}.npy")
    config.embeds_mean = config.embeds_mean.flatten()
    print(config.embeds_mean.shape)
    
# run search_algo and save results
config_name = f"mcts_embeds--level-{level}--{config.version}--n-{config.n}--d-{config.max_depths}--b-{config.step_budget:03d}--lam-{config.lam}--dalpha-{config.ds_alpha}--dbeta-{config.ds_beta}--ppl-{config.use_ppl}--normalize-{config.embeds_normalizing}--mcenter-{config.embeds_centering}"
print(config_name)
result_dir = f"results/mcts_embeds--level-{level}/{config_name}"
try:
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created successfully.")
except FileExistsError:
    print(f"Directory '{result_dir}' already exists.")
except OSError as e:
    print(f"Error creating directory: {e}")
    stop

num_trials = 5
/groups/kjun/tnn/datasets//prm800k/math_splits1
num_questions = 2
(2048,)
mcts_embeds--level-4--t811--n-4--d-20--b-040--lam-0.01--dalpha-100.0--dbeta-1.0--ppl-True--normalize-True--mcenter-True
Directory 'results/mcts_embeds--level-4/mcts_embeds--level-4--t811--n-4--d-20--b-040--lam-0.01--dalpha-100.0--dbeta-1.0--ppl-True--normalize-True--mcenter-True' already exists.


In [17]:
start_time1 = time.time()
for trial_idx in range(num_trials):
    print(f"trial {trial_idx}")
    start_time = time.time()
    
    # best_of_n(batch_of_questions, config, llm_vllm, random_seeds[trial_idx])
    results = mcts_embeds_search_v03_02_00._search(batch_of_questions, config, trial_idx, llm_vllm, llm_vllm_embeds, prm)
    with open(f"{result_dir}/generate_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
        json.dump(results, fout)
        fout.write('\n')

    # compute the time
    if trial_idx % 1 == 0:
        total_time = time.time() - start_time
        # time_per_trial = total_time/(trial_idx+1)
        time_per_trial = total_time
        time_per_question = time_per_trial/num_questions
        # print(f"trial {trial_idx}")
        print(f"it takes {time_per_question:0.4f}s per question")
        print(f"it takes {time_per_trial/3600:0.2f}h per trial")

    add_completions_to_dataset(result_dir, config_name, dataset, prm, trial_idx, config)


-> p = 0

-> d = 0


trial 0


0.1
   q-value = 0.1968
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.2
   q-value = 0.2069
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.3
   q-value = 0.1755
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.4
   q-value = 0.4766
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
selected_node = 0.4
step_cnt = 1

-> d = 1
0.4.1
   q-value = 0.7124
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.4.2
   q-value = 0.6040
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.4.3
   q-value = 0.7310
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.4.4
   q-value = 0.4456
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
selected_node = 0.4.3
step_cnt = 2

-> d = 2
0.4.3.1
   q-value = 0.8081
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.4.3.2
   q-value = 0.7661
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.4.3.3
   q-val

it takes 75.2415s per question
it takes 0.04h per trial


NameError: name 'add_completions_to_dataset' is not defined